# Tracking 
## Single molecule tracking pipeline, python version
This script will detect the emitters in h5 file, and link the localizations into tracks using trackpy. It will return the D* distributions for one or several fields of view derived form an experiment. To proceed, run the cells below by clicking the 'Play' button on the upper panel or select the cell and press Shift+Enter.


In [ ]:
%load_ext dotenv
%dotenv config.env

In [ ]:
%load_ext autoreload 
%autoreload 2

In [ ]:
# Import the dependencies.
# (might take few min)

import random
import h5py
import numpy as np
import pandas as pd
import seaborn as sns
import os
import subprocess
import uuid
from pathlib import Path
import ipywidgets as widgets
import trackpy as tp

from io import BytesIO
from matplotlib import pyplot as plt
from math import log
from ipywidgets import Button, Layout, interactive, fixed
from ipyfilechooser import FileChooser

from utils.logging import setup_logger, log_and_print
from utils.conditions_to_dict import conditions_to_df, get_fovs
from utils.tiff_writer import download_img_allas
from utils.locs_trackpy import get_locs_trackpy, h5_get_locs_preview
from utils.track_trackpy import track_py, plot_tracks, locs_preview_contours
from utils.get_features import get_Dapp
from utils.tiff_writer import show_mask
from utils.align_mask import manual_reg_plot
from utils.csv_writer import show_df_style
from utils.plot import *
from utils.set_widgets import *
from datetime import datetime

from utils.tp_refine_com_arr_monkey_patch import refine_com_arr_mp, refine_com_mp
tp.refine.center_of_mass.refine_com_arr = refine_com_arr_mp
tp.refine.center_of_mass.refine_com = refine_com_mp
tp.refine.refine_com = refine_com_mp
tp.feature.refine_com = refine_com_mp         



import warnings
warnings.filterwarnings('ignore')


## Selecting and downloading the data

In [ ]:
# Select the aoutput directory to save plots, logs and data 

path = Path(os.getenv('SMPP_INPUT_DIR'))
output_for_data = FileChooser(path)
output_for_data.show_only_dirs = True
display(output_for_data)


In [ ]:
# setup logging 

analysis_id = str(uuid.uuid1())[:8]
git_hash = subprocess.check_output(['git', 'rev-parse', 'HEAD']).decode('ascii').strip()

logger = setup_logger(output_for_data, analysis_id)
log_and_print(f'output directory: {output_for_data.selected_path}')
log_and_print(f'analysis_id_short:{analysis_id}')
log_and_print(f'git_hash:{git_hash}')


In [ ]:
# Click 'Play' to search across the database and choose the experiment which you'd like to analyse.
# This script will only work with the experiments performed on PyME gui after 27.02.2025

# Only one experiment entry is allowed to be processed.
# Use the 'clear' function in the 'date' field to remove any unwanted entry.
# It is ok to use a comma and space to separate the row numbers in the 'exclude' and 'include' fields.
# Keep all the fields intact to view the entire database (though this may become inconvenient soon)

database = pd.read_csv(os.getenv('SMPP_DATABASE'), sep = '\t', dtype = 'str')
fields_list, fld_widgets_list, box = filter_data(database)
display(box)


In [ ]:
# Run this cell when ready with the selection above,
# check if the experiment you choose is the one you want.

experiment = conditions_to_df(database, fld_widgets_list, fields_list, 'Experiments')
experiment


In [ ]:
# Run this cell to see selected imaging files

exp_id = experiment['uuid_experiment'].values[0]
files, masks = get_fovs(database, exp_id)

log_and_print('\nSelected files:')
show_df_style(files)
log_and_print('\n\nSelected masks:')
show_df_style(masks)
log_and_print('')


In [ ]:
# Select the FOVs you'd like to process (must have a mask associated, see 'masks:' field)

box, output, checked_ids = fov_all_selector(files, masks)
log_and_print('')
display(box, output)
log_and_print('')


In [ ]:
# Show selected FOVs

files_selected = files[files['uuid_fov'].isin(checked_ids)]
masks_selected = masks[masks['uuid_fov'].isin(checked_ids)]

log_and_print(f"\n\n{files_selected.shape[0]} files selected, {masks_selected.shape[0]} masks\n")
show_df_style(files_selected)
log_and_print('')


In [ ]:
# Get the tracking data from Allas (only the tracking channel for each FOV will be shown)
# (might take few min)

im_data = []
fluos = []

for file in files_selected['uuid_file'].unique():
    path = eval(files.loc[files['uuid_file']==file, 'path'].item())
    h5 = h5py.File(BytesIO(download_img_allas(path[0], path[1])), 'r')
    if h5['ImageData'].shape[0] > 10**4:

        fov_id = files.loc[files['uuid_file']==file, 'uuid_fov'].item()
        
        try:
            p_mask = masks.loc[masks['uuid_fov']==fov_id, 'path'].item()
            mask_uuid = masks.loc[masks['uuid_fov']==fov_id, 'uuid_analysis'].item()
            
            im_data.append((file, path, p_mask, mask_uuid))
            fluos.append(h5['ImageData'])
            log_and_print(f"{path[1]} downloaded")
            
        except ValueError:
            log_and_print(f'can\'t find mask for file {path[1]}')

print('All files are ready')


## Tracking

In [ ]:
# Run this cell and set a threschold for trackpy locs detection
# matlab threshold:  matlab-like bandpass

w = h5_w_tracking_params()
display(w)


In [ ]:
# Specify matlab thresholds for different FOVs if needed

if not w.children[4].value:
    mw = h5_w_multiple_ml_thresholds(im_data, w.children[1].value)
    display(mw)
    

In [ ]:
# Set the thresholds 

if w.children[4].value:
    thresholds = [w.children[1].value]*len(im_data)   

else: 
    thresholds = [wdg.value for wdg in mw.children]
    

In [ ]:
# Get time between frames from experiment meta
# Show chosen tracking params

tp_threshold = w.children[0].value




import xml.etree.ElementTree as ET
xml = ET.parse(experiment['path'].item())
# xml = ET.parse(os.getenv('SMPP_DATA_DIR')+'/meta_xml/250517_AW_Ec_136_M9glyCAAT_JFX554_37b8b238.xml')
root = xml.getroot()

# getting the short name for results file
xml_file_name = experiment['path'].item().split('/')[-1]
exp_name = xml_file_name.split('.')[0]
exp_name_short = '_'.join([exp_name.split('_')[0], 
                           exp_name.split('_')[1], 
                           exp_name.split('_')[-1]])

log_and_print(exp_name_short)
pixel = float(root[5][0].text)
if pixel>1:
    pixel = pixel/1000 # for old h5 files where voxelsize is in nm
log_and_print(f'\npixel: {round(pixel, 3)} micron')
dT = float(root[5][1][0].text)
log_and_print(f'dT: {round(dT*1000, 2)} ms')
log_and_print(f'thresholds used:\n'
     f'trackpy threshold: {tp_threshold}\n'
     f'matlab-like threshold: {thresholds}'
     )

# preview the frames

for im, fluo, ml_threshold in zip(im_data, fluos, thresholds):
    print('')
    log_and_print(im[1][1])
    print('')
    h5_get_locs_preview(fluo, tp_threshold, 
                        ml_threshold, 
                        prev_frame=int(w.children[3].value), 
                        gauss_short=1, 
                        boxcar_long=7)


In [ ]:
# This cell will generate the dataframe with localizations predictedd by trackpy
# trackpy.batch can be parallelized, so more CPU cores = faster prediction

# trackpy details:
# https://github.com/soft-matter/trackpy
# http://doi.org/10.1006/jcis.1996.0217


all_fovs_locs = []
for im, fluo, ml_threshold in zip(im_data, fluos, thresholds):       
    fluo_bytes = fluo
    log_and_print(im)
    locs, proj = get_locs_trackpy(fluo_bytes, tp_threshold, ml_threshold, gauss_short=1, boxcar_long=7, h5=True)
    
    all_fovs_locs.append((locs, proj, im[2], im[0], im[3]))
    # locs df, sd projection, mask file, tiff UUID, mask UUID
    

for loc in all_fovs_locs:
    log_and_print(f'\nLocalizations for fov {loc[2]}:\n'
          f'locs dataframe shape: {loc[0].shape}\n'
          f'SD projection shape (sanity): {loc[1].shape}\n'
          f'tiff UUID: {loc[3]}\n'
          f'mask UUID: {loc[4]}\n')

log_and_print(f'All ({len(all_fovs_locs)}) FOVs are ready\n'
      f'Proceed with tracking (next cell)')


In [ ]:
# Run this cell if you chose manual mask alignment

offsets = {}
for meta in im_data:
    offsets[meta[2].split('/')[-1]]=(0, 0)

if w.children[2].value == 'manual':
    fw, xw, yw, xs, ys = manual_reg_widgets(all_fovs_locs)
    interactive_plot = interactive(manual_reg_plot, 
                                   xlim=xw, ylim=yw, 
                                   xshift=xs, yshift=ys, 
                                   fov_n=fw, 
                                   fovs_list=fixed(all_fovs_locs),
                                   offsets_tuple=fixed((offsets,))
                                  )
    output = interactive_plot.children[-1]
    output.layout.height = '600px'
    display(interactive_plot)
    

In [ ]:
# Run this cell if you want to specify frame range to use for analysis

if not w.children[5].value:
  
    ranges = {}
    
    fw, frame_range = m_frame_range_wds(all_fovs_locs)
    interactive_plot = interactive(manual_frame_range, 
                                   frame_range=frame_range,
                                   fov_n=fw, 
                                   fovs_list=fixed(all_fovs_locs),
                                   ranges_tuple=fixed((ranges,))
                                   )
    output = interactive_plot.children[-1]
    output.layout.height = '600px'
    display(interactive_plot)
    



In [ ]:
# data cleaning

# if frame range is specified manually
if not w.children[5].value:
    log_and_print(ranges)
    for idx, loc in enumerate(all_fovs_locs):
        log_and_print(f'locs df shape from {loc[2]}:')
        log_and_print(f'before cropping: {loc[0].shape}')
        locs_df = loc[0]
        locs_df = locs_df[locs_df['frame']>=ranges[loc[2]][0]]
        locs_df = locs_df[locs_df['frame']<=ranges[loc[2]][1]]
        all_fovs_locs[idx] = (locs_df, loc[1], loc[2], loc[3], loc[4])
        log_and_print(f'after cropping: {all_fovs_locs[idx][0].shape}')

    
# automatic outliers removal
if w.children[6].value:
    for idx, loc in enumerate(all_fovs_locs):
        log_and_print(f'locs df shape from {loc[2]}:')
        log_and_print(f'before removing outliers: {loc[0].shape}')

        locs_df = loc[0]

        bounds = {}
        for col in ['mass', 'size', 'signal', 'raw_mass', 'ecc', 'ep']:
            col_mean=locs_df[col].mean()
            lower_bound = col_mean-4*locs_df[col].std()
            upper_bound = col_mean+4*locs_df[col].std()
            bounds[col] = (lower_bound, upper_bound)
                    
        filter_mask = True
        for col in ['mass', 'size', 'signal', 'raw_mass']:
            lower_bound, upper_bound = bounds[col]
            filter_mask &= (locs_df[col] >= lower_bound) & (locs_df[col] <= upper_bound)
        locs_df = locs_df[filter_mask]

        locs_df = locs_df[(locs_df['ecc'] > 0) & (locs_df['ecc'] < 1)] # by ecc definition

        filter_mask = True
        for col in ['ecc', 'ep']:
            lower_bound, upper_bound = bounds[col]
            filter_mask &= (locs_df[col] <= upper_bound)
        locs_df = locs_df[filter_mask]        
        
        
        all_fovs_locs[idx] = (locs_df, loc[1], loc[2], loc[3], loc[4])
        log_and_print(f'after removing outliers: {all_fovs_locs[idx][0].shape}')
        display(all_fovs_locs[idx][0].describe())

    

In [ ]:
# Run this cell to link the localizations into tracks
# trackpy code is used here

# trackpy details:
# https://github.com/soft-matter/trackpy
# http://doi.org/10.1006/jcis.1996.0217

memory = 1 if w.children[7].value else 0
tracks = []

for loc in all_fovs_locs:
    tracks.append(track_py(loc[0], loc[1], loc[2], loc[3], loc[4], 
                           flip_mask=False, 
                           allow_missing=memory, 
                           tp_link_strategy='drop', 
                           align_m=w.children[2].value)) 

log_and_print(f'All ({len(all_fovs_locs)}) localisations are processed\n'
      f'Proceed with plotting to preview the results and save the data')


In [ ]:
# Combine the data from separate FOVs into a single dataframe
# Preview the tracks

all_tracks = pd.concat(tracks)
# all_tracks.drop('frame', axis=1, inplace=True)

for i in all_tracks['tiff_uuid'].unique():
    all_tracks.loc[(all_tracks['tiff_uuid']==i), 'comment']=files.loc[(files['uuid_file']==i), 'comment'].values[0]

tracks_gb = all_tracks.groupby(['mask_uuid', 'cell_id_int', 'particle'])
cells_gb  = all_tracks.groupby(['mask_uuid', 'cell_id_int'])
fovs_gb   = all_tracks.groupby('mask_uuid')




In [ ]:
# Get aggregated data

tracks_agg = tracks_gb.agg({'frame':np.ptp, 'comment':'first'}).rename(columns={'frame':'track_length'})
tracks_agg['Dapp'] = tracks_gb.apply(get_Dapp, dT = dT, pixel=pixel)
tracks_agg.reset_index(inplace=True)

tracks_agg['Dapp_log']=np.log10(tracks_agg['Dapp'])
tracks_agg['mask_uuid_s']=tracks_agg['mask_uuid'].str[:8]
# display(tracks_agg.head())

all_tracks = all_tracks.set_index(['mask_uuid', 'cell_id_int', 'particle'])
tracks_agg = tracks_agg.set_index(['mask_uuid', 'cell_id_int', 'particle'])
all_tracks.loc[:,['track_length', 'Dapp', 'Dapp_log']] = tracks_agg.loc[:,['track_length', 'Dapp', 'Dapp_log']]

# all_tracks.columns



In [ ]:
# Plotting

m_directory = os.getenv('SMPP_DATA_DIR')+'/analysis/masks_omni/'

# choose plots
ridges_side_by_side(all_tracks, output_for_data, width=14, as_svg=False)
plot_D_overlay(tracks_agg)
plot_track_length(tracks_agg)
plot_Dapp_vs_tlength(tracks_agg)
plot_all_locs(all_tracks, m_directory)
plot_slow_fast(all_tracks, m_directory)



In [ ]:
# Dump the data

directory = output_for_data.selected_path
out_path = f'{directory}/tracks_exp_{exp_id[:8]}_a_{analysis_id}.csv'
all_tracks.to_csv(out_path)

log_and_print('Data saved to:')
log_and_print(out_path)


In [ ]:
# That's it, you can close the notebook, or shut down the interactive session.
